# DIFF (2026-03-21): `lb-35-9-ensembling-post-processing-baseline.ipynb` をコピーして `notebooks/006/[1]final-submission-v1.ipynb` を作成

- 対象: `[1]final-submission-v1.ipynb`
- 元: `lb-35-9-ensembling-post-processing-baseline.ipynb`
- 種別: 新系統の作成
- 変更点:
  - モデルを ByT5-large / ByT5-base / ByT5-small の3本に変更
  - `max_input_length=1024`, `max_new_tokens=1024` に変更
  - フォールバックを `"<gap>"` に変更
  - 前処理/後処理を `notebooks/002/[4-8]submit-notebook-v1.ipynb` と同一ロジックに合わせる
  - OOM 発生時のログを明確化
- 変更理由/仮説: 3サイズの ByT5 を束ね、長文を切り詰めにくくしたうえで MBR の候補多様性を増やす


# **Author: Cho Royou**

## 1. Imports and runtime setup
This section loads all required libraries and sets a few runtime defaults for Kaggle.

In [1]:
import os
import gc
import re
import json
import math
import random
import logging
import warnings

from pathlib import Path
from dataclasses import dataclass
from contextlib import nullcontext
from typing import List, Tuple

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

'true'

## 2. BF16 utility helpers
These helpers detect whether BF16 autocast is available and create the proper context manager.

In [2]:
def _cuda_bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except Exception:
        return False


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda" and _cuda_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def _is_cuda_kernel_image_error(exc: BaseException) -> bool:
    msg = str(exc).lower()
    return (
        "no kernel image is available for execution on the device" in msg
        or "cudaerrornokernelimagefordevice" in msg
    )


## 3. Configuration
3モデル構成、長めの入出力長、OOM が分かるログ出力に合わせて設定をまとめる。

In [3]:
@dataclass
class EnsembleMBRConfig:
    test_data_path: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    output_dir: str = "/kaggle/working/"
    model_paths: Tuple[str, ...] = (
        "/kaggle/input/datasets/tatsuokoshida/dpc-byt5-large-v002-6",
        "/kaggle/input/datasets/tatsuokoshida/dpc-byt5-base-v002-6",
        "/kaggle/input/notebooks/tatsuokoshida/2-8-dpc-starter-train-v1/byt5-akkadian-model/fulltrain_byt5-small_multitask",
    )
    model_labels: Tuple[str, ...] = (
        "ByT5-large",
        "ByT5-base",
        "ByT5-small",
    )

    max_input_length: int = 1024
    max_new_tokens: int = 1024
    batch_size: int = 1
    num_workers: int = 2
    num_buckets: int = 6

    num_beam_cands: int = 4
    num_beams: int = 8
    length_penalty: float = 1.3
    early_stopping: bool = True
    repetition_penalty: float = 1.2

    num_sample_cands: int = 2
    mbr_top_p: float = 0.92
    mbr_temperature: float = 0.75
    mbr_pool_cap: int = 32

    use_mixed_precision: bool = False
    use_better_transformer: bool = False
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True
    checkpoint_freq: int = 200
    attn_implementation: str = "eager"

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)

        if len(self.model_paths) != len(self.model_labels):
            raise ValueError("model_paths and model_labels must have the same length")

        if not torch.cuda.is_available():
            self.use_mixed_precision = False
            self.use_better_transformer = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _cuda_bf16_supported()
        )

        assert self.num_beams >= self.num_beam_cands


## 4. Logging
Logs are written to both stdout and a file inside the working directory.

In [4]:
def setup_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(exist_ok=True, parents=True)

    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "ensemble_mbr.log"),
        ],
    )
    return logging.getLogger("ensemble_mbr")

## 5. Preprocessing rules
This block normalizes transliterations before feeding them into the models.

### Included transformations
- ASCII transliteration to diacritics
- Determinative handling
- Gap normalization
- Character cleanup
- Fraction and decimal normalization

In [5]:
# DIFF (2026-03-21): 前処理/後処理は `[2-8]dpc-starter-train-v1.ipynb` と同一ロジック前提（submit 側は build_inputs を利用）
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List


PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


## 6. Postprocessing rules
This block standardizes model outputs before MBR selection.

### Included transformations
- Gap unification
- `PN` handling
- Commodity-specific normalization
- Shekel fraction normalization
- Grammar-marker removal
- Month normalization
- Cleanup of repeated fragments and whitespace

In [6]:
_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()

def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Competition input directory not found. Tried: " + ", ".join(input_dirs)
    )


def build_inputs(test_df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    if "id" not in test_df.columns:
        raise ValueError(f"test.csv must contain 'id'. columns={list(test_df.columns)}")
    ids = test_df["id"].astype(str).tolist()

    if "task" in test_df.columns:
        tasks = test_df["task"].astype(str).tolist()
    else:
        tasks = ["akk2en"] * len(test_df)

    if "source" in test_df.columns:
        sources = test_df["source"].astype(str).tolist()
    elif "transliteration" in test_df.columns:
        sources = test_df["transliteration"].astype(str).tolist()
    else:
        raise ValueError(
            "test.csv must contain either 'source' or 'transliteration'. "
            f"columns={list(test_df.columns)}"
        )

    inputs: list[str] = [""] * len(sources)

    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    src_akk = [sources[i] for i in idx_akk2en]
    src_en = [sources[i] for i in idx_en2akk]

    src_akk_pp = _preprocessor.preprocess_batch(src_akk) if src_akk else []
    src_en_pp = _postprocessor.postprocess_batch(src_en) if src_en else []

    for i, s_norm in zip(idx_akk2en, src_akk_pp):
        inputs[i] = PREFIX_AKK_EN + s_norm
    for i, s_norm in zip(idx_en2akk, src_en_pp):
        inputs[i] = PREFIX_EN_AKK + s_norm

    # Keep strict: unknown tasks are a hard error.
    unknown = sorted(set(tasks) - {"akk2en", "en2akk"})
    if unknown:
        raise ValueError(f"Unknown task(s): {unknown}")

    return ids, tasks, inputs


## 7. Dataset and bucketed batching
The dataset applies preprocessing once. The custom sampler groups similar sequence lengths together to reduce padding waste.

In [7]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: OptimizedPreprocessor, logger: logging.Logger):
        self.sample_ids = df["id"].tolist()
        proc = preprocessor.preprocess_batch(df["transliteration"].tolist())
        self.input_texts = ["translate Akkadian to English: " + t for t in proc]
        logger.info(f"Dataset: {len(self.sample_ids)} samples")

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        return self.sample_ids[idx], self.input_texts[idx]


class BucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets, logger, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle

        lengths = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])

        bsize = max(1, len(sorted_idx) // max(1, num_buckets))
        self.buckets = [
            sorted_idx[i * bsize : None if i == num_buckets - 1 else (i + 1) * bsize]
            for i in range(num_buckets)
        ]

        for i, b in enumerate(self.buckets):
            if b:
                bl = [lengths[x] for x in b]
                logger.info(f"  Bucket {i}: {len(b)} samples, len [{min(bl)}, {max(bl)}]")

    def __iter__(self):
        for bucket in self.buckets:
            b = list(bucket)
            if self.shuffle:
                random.shuffle(b)
            for i in range(0, len(b), self.batch_size):
                yield b[i:i+self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(b) / self.batch_size) for b in self.buckets)

## 8. Model wrapper
This wrapper handles:
- tokenizer/model loading
- optional BetterTransformer conversion
- candidate generation
- safe unload and VRAM cleanup

In [8]:
class ModelWrapper:
    def __init__(self, model_path: str, cfg: EnsembleMBRConfig, logger: logging.Logger, label: str):
        self.cfg = cfg
        self.logger = logger
        self.label = label
        self.model_path = model_path
        self.tokenizer = None
        self.model = None
        self.device = cfg.device
        try:
            self._load(self.device)
        except Exception as e:
            if self.device.type == "cuda" and _is_cuda_kernel_image_error(e):
                self.logger.warning(f"[{self.label}] CUDA kernel incompatibility detected during load: {e}")
                self._load(torch.device("cpu"))
            else:
                raise

    def _load(self, device: torch.device):
        self.device = device
        self.logger.info(f"[{self.label}] Loading from {self.model_path} on {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_path,
            attn_implementation=self.cfg.attn_implementation,
        ).to(self.device).eval()

        if self.device.type == "cuda":
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass
            self.logger.info(f"[{self.label}] attention backend: {self.cfg.attn_implementation}")

        n = sum(p.numel() for p in self.model.parameters())
        self.logger.info(f"[{self.label}] {n:,} parameters")

        if self.device.type == "cuda":
            used = torch.cuda.memory_allocated() / 1e9
            self.logger.info(f"[{self.label}] GPU mem used: {used:.2f} GB")

        if self.cfg.use_better_transformer and self.device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                self.logger.info(f"[{self.label}] BetterTransformer applied")
            except Exception as e:
                self.logger.warning(f"[{self.label}] BetterTransformer skipped: {e}")
        elif self.device.type == "cuda":
            self.logger.info(f"[{self.label}] BetterTransformer disabled to avoid GPU kernel incompatibility")

    def collate(self, batch_samples):
        ids = [s[0] for s in batch_samples]
        texts = [s[1] for s in batch_samples]

        enc = self.tokenizer(
            texts,
            max_length=self.cfg.max_input_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return ids, enc

    def generate_candidates(self, input_ids, attention_mask, beam_size: int) -> List[List[str]]:
        cfg = self.cfg
        B = input_ids.shape[0]
        ctx = _bf16_ctx(self.device, cfg.use_bf16_amp)

        with ctx:
            nb = max(beam_size, cfg.num_beam_cands)
            beam_out = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                do_sample=False,
                num_beams=nb,
                num_return_sequences=cfg.num_beam_cands,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,
                use_cache=True,
            )
            beam_texts = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)

            samp_texts = []
            if cfg.num_sample_cands > 0:
                samp_out = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=True,
                    num_beams=1,
                    top_p=cfg.mbr_top_p,
                    temperature=cfg.mbr_temperature,
                    num_return_sequences=cfg.num_sample_cands,
                    max_new_tokens=cfg.max_new_tokens,
                    repetition_penalty=cfg.repetition_penalty,
                    use_cache=True,
                )
                samp_texts = self.tokenizer.batch_decode(samp_out, skip_special_tokens=True)

        Rb, Rs = cfg.num_beam_cands, cfg.num_sample_cands
        pools = []
        for i in range(B):
            p = list(beam_texts[i * Rb:(i + 1) * Rb])
            if Rs > 0:
                p += list(samp_texts[i * Rs:(i + 1) * Rs])
            pools.append(p)

        return pools

    def reload_on_cpu(self):
        if self.device.type == "cpu":
            return
        self.logger.warning(f"[{self.label}] Falling back to CPU due to CUDA kernel incompatibility")
        self.unload(skip_cuda_sync=True)
        self._load(torch.device("cpu"))

    def unload(self, skip_cuda_sync: bool = False):
        label = self.label

        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except Exception:
            pass

        if self.model is not None:
            del self.model
        if self.tokenizer is not None:
            del self.tokenizer
        self.model = None
        self.tokenizer = None

        gc.collect()
        if torch.cuda.is_available() and not skip_cuda_sync:
            try:
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
                free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
                self.logger.info(f"[{label}] Unloaded. GPU free: {free:.2f} GB")
            except Exception as e:
                self.logger.warning(f"[{label}] GPU cleanup skipped: {e}")


## 9. MBR selector
MBR chooses the candidate with the highest average chrF++ agreement against the other candidates in the pool.

In [9]:
class MBRSelector:
    def __init__(self, pool_cap: int = 32):
        self._metric = sacrebleu.metrics.CHRF(word_order=2)
        self.pool_cap = pool_cap

    def _chrfpp(self, a: str, b: str) -> float:
        a, b = (a or "").strip(), (b or "").strip()
        if not a or not b:
            return 0.0
        return float(self._metric.sentence_score(a, [b]).score)

    @staticmethod
    def _dedup(xs: List[str]) -> List[str]:
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    def pick(self, candidates: List[str]) -> str:
        cands = self._dedup(candidates)

        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        best_i, best_s = 0, -1e9
        for i in range(n):
            s = sum(self._chrfpp(cands[i], cands[j]) for j in range(n) if j != i)
            s /= max(1, n - 1)
            if s > best_s:
                best_s, best_i = s, i

        return cands[best_i]

## 10. End-to-end ensemble engine
This class runs the full pipeline:

1. preprocess test inputs  
2. generate candidates with Model A  
3. generate candidates with Model B  
4. merge candidate pools  
5. postprocess outputs  
6. select the final translation with MBR  
7. write checkpoints and validate the submission

In [10]:
class EnsembleMBREngine:
    def __init__(self, cfg: EnsembleMBRConfig, logger: logging.Logger):
        self.cfg = cfg
        self.logger = logger
        self.preprocessor = OptimizedPreprocessor()
        self.postprocessor = VectorizedPostprocessor()
        self.mbr = MBRSelector(pool_cap=cfg.mbr_pool_cap)

    def _adaptive_beams(self, attn: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            return self.cfg.num_beams

        med = float(attn.sum(dim=1).float().median().item())
        short = max(self.cfg.num_beam_cands, self.cfg.num_beams // 2)
        return short if med < 100 else self.cfg.num_beams

    def _build_dataloader(self, dataset: AkkadianDataset, wrapper: ModelWrapper) -> DataLoader:
        if self.cfg.use_bucket_batching:
            sampler = BucketBatchSampler(
                dataset,
                self.cfg.batch_size,
                self.cfg.num_buckets,
                self.logger,
            )
            return DataLoader(
                dataset,
                batch_sampler=sampler,
                num_workers=self.cfg.num_workers,
                collate_fn=wrapper.collate,
                pin_memory=(wrapper.device.type == "cuda"),
            )

        return DataLoader(
            dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            collate_fn=wrapper.collate,
            pin_memory=(wrapper.device.type == "cuda"),
        )

    def _run_one_model_once(self, wrapper: ModelWrapper, dataset: AkkadianDataset) -> dict[str, List[str]]:
        dl = self._build_dataloader(dataset, wrapper)
        pools_by_id: dict[str, List[str]] = {}

        with torch.inference_mode():
            for batch_ids, enc in tqdm(dl, desc=f"[{wrapper.label}]"):
                input_ids = enc.input_ids.to(wrapper.device, non_blocking=(wrapper.device.type == "cuda"))
                attn = enc.attention_mask.to(wrapper.device, non_blocking=(wrapper.device.type == "cuda"))
                beam_size = self._adaptive_beams(attn)

                try:
                    batch_pools = wrapper.generate_candidates(input_ids, attn, beam_size)
                    for sid, pool in zip(batch_ids, batch_pools):
                        pools_by_id[str(sid)] = pool
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error(
                            f"OOM detected in [{wrapper.label}] for sample_ids={list(batch_ids)}; "
                            f"batch_size={len(batch_ids)} beam_size={beam_size}. Skipping batch."
                        )
                        if wrapper.device.type == "cuda" and torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        for sid in batch_ids:
                            pools_by_id.setdefault(str(sid), [])
                    else:
                        raise
                except Exception as e:
                    self.logger.error(f"[{wrapper.label}] batch error for sample_ids={list(batch_ids)}: {e}")
                    for sid in batch_ids:
                        pools_by_id.setdefault(str(sid), [])

                if wrapper.device.type == "cuda" and torch.cuda.is_available():
                    torch.cuda.empty_cache()

        return pools_by_id

    def _run_one_model(self, wrapper: ModelWrapper, dataset: AkkadianDataset) -> dict[str, List[str]]:
        try:
            return self._run_one_model_once(wrapper, dataset)
        except Exception as e:
            if wrapper.device.type == "cuda" and _is_cuda_kernel_image_error(e):
                self.logger.warning(f"[{wrapper.label}] CUDA kernel incompatibility detected: {e}")
                wrapper.reload_on_cpu()
                return self._run_one_model_once(wrapper, dataset)
            raise

    def run(self, test_df: pd.DataFrame) -> pd.DataFrame:
        cfg, logger = self.cfg, self.logger

        logger.info("=" * 60)
        logger.info("Ensemble x MBR | Cross-model candidate pooling")
        logger.info(f"  Models        : {len(cfg.model_paths)}")
        for label, model_path in zip(cfg.model_labels, cfg.model_paths):
            logger.info(f"  {label:12s}: {model_path}")
        logger.info(f"  Pool per model: beam x {cfg.num_beam_cands} + sample x {cfg.num_sample_cands}")
        logger.info(f"  MBR pool cap  : {cfg.mbr_pool_cap}")
        logger.info(f"  BF16 AMP      : {cfg.use_bf16_amp}")
        logger.info(f"  batch_size    : {cfg.batch_size}")
        logger.info("=" * 60)

        dataset = AkkadianDataset(test_df, self.preprocessor, logger)
        sample_ids = [str(s) for s in dataset.sample_ids]
        model_pools: List[dict[str, List[str]]] = []

        total_models = len(cfg.model_paths)
        for idx, (label, model_path) in enumerate(zip(cfg.model_labels, cfg.model_paths), start=1):
            logger.info(f"Phase {idx}/{total_models + 1} - {label} inference")
            wrapper = ModelWrapper(model_path, cfg, logger, label)
            pools = self._run_one_model(wrapper, dataset)
            model_pools.append(pools)
            wrapper.unload(skip_cuda_sync=(wrapper.device.type != "cuda"))
            del wrapper

        logger.info(f"Phase {total_models + 1}/{total_models + 1} - Pool merge + MBR selection")
        results: List[Tuple[str, str]] = []

        for sid in tqdm(sample_ids, desc="MBR"):
            combined: List[str] = []
            for pools in model_pools:
                combined.extend(pools.get(sid, []))
            pp = self.postprocessor.postprocess_batch(combined) if combined else []
            chosen = self.mbr.pick(pp)

            if not chosen or not chosen.strip():
                self.logger.warning(f"Fallback applied for sample_id={sid}: empty candidate pool after postprocessing")
                chosen = "<gap>"

            results.append((sid, chosen))

            if len(results) % cfg.checkpoint_freq == 0:
                ckpt = Path(cfg.output_dir) / f"checkpoint_{len(results)}.csv"
                pd.DataFrame(results, columns=["id", "translation"]).to_csv(ckpt, index=False)
                logger.info(f"  Checkpoint: {len(results)} rows -> {ckpt}")

        result_df = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(result_df)
        return result_df

    def _validate(self, df: pd.DataFrame):
        logger = self.logger
        logger.info("=" * 60)

        empty = df["translation"].str.strip().eq("").sum()
        lens = df["translation"].str.len()

        logger.info(f"Empty     : {empty} ({100 * empty / max(1, len(df)):.2f}%)")
        logger.info(
            f"Len mean  : {lens.mean():.1f}  median: {lens.median():.1f}  min: {lens.min()}  max: {lens.max()}"
        )

        for idx in [0, len(df) // 4, len(df) // 2, 3 * len(df) // 4, len(df) - 1]:
            row = df.iloc[idx]
            logger.info(f"  ID {row['id']}: {str(row['translation'])[:80]}")

        logger.info("=" * 60)


## 11. Runtime summary helper
This small helper prints the environment before inference starts.

In [11]:
def print_env(cfg: EnsembleMBRConfig):
    print(f"PyTorch  : {torch.__version__}")
    print(f"CUDA     : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU      : {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Mem  : {mem:.2f} GB")
        print(f"BF16     : {_cuda_bf16_supported()}")

    print(f"BF16 AMP : {cfg.use_bf16_amp}")
    print(f"Attention: {cfg.attn_implementation}")


## 12. Run inference and create `submission.csv`
This final block executes the full pipeline and saves:
- `submission.csv`
- `ensemble_mbr.log`
- `ensemble_mbr_config.json`

In [12]:
cfg = EnsembleMBRConfig()
logger = setup_logging(cfg.output_dir)

print_env(cfg)

logger.info(f"Loading test data: {cfg.test_data_path}")
test_df = pd.read_csv(cfg.test_data_path, encoding="utf-8")
logger.info(f"Test samples: {len(test_df)}")

engine = EnsembleMBREngine(cfg, logger)
results_df = engine.run(test_df)

out_path = Path(cfg.output_dir) / "submission.csv"
results_df.to_csv(out_path, index=False)
logger.info(f"Saved -> {out_path} ({len(results_df)} rows)")

cfg_snap = {
    "model_paths": list(cfg.model_paths),
    "model_labels": list(cfg.model_labels),
    "num_beam_cands": cfg.num_beam_cands,
    "num_beams": cfg.num_beams,
    "num_sample_cands": cfg.num_sample_cands,
    "length_penalty": cfg.length_penalty,
    "mbr_top_p": cfg.mbr_top_p,
    "mbr_temperature": cfg.mbr_temperature,
    "mbr_pool_cap": cfg.mbr_pool_cap,
    "max_input_length": cfg.max_input_length,
    "max_new_tokens": cfg.max_new_tokens,
    "repetition_penalty": cfg.repetition_penalty,
    "use_bf16_amp": cfg.use_bf16_amp,
    "batch_size": cfg.batch_size,
    "attn_implementation": cfg.attn_implementation,
}

with open(Path(cfg.output_dir) / "ensemble_mbr_config.json", "w") as f:
    json.dump(cfg_snap, f, indent=2)

print("\n" + "=" * 60)
print("Ensemble x MBR complete")
print(f"Submission : {out_path}")
print(f"Total rows : {len(results_df)}")
print("=" * 60)


2026-03-21 11:37:19,093 | INFO | Loading test data: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
2026-03-21 11:37:19,105 | INFO | Test samples: 4
2026-03-21 11:37:19,106 | INFO | ============================================================
2026-03-21 11:37:19,106 | INFO | Ensemble x MBR | Cross-model candidate pooling
2026-03-21 11:37:19,107 | INFO |   Models        : 3
2026-03-21 11:37:19,108 | INFO |   ByT5-large  : /kaggle/input/datasets/tatsuokoshida/dpc-byt5-large-v002-6
2026-03-21 11:37:19,109 | INFO |   ByT5-base   : /kaggle/input/datasets/tatsuokoshida/dpc-byt5-base-v002-6
2026-03-21 11:37:19,109 | INFO |   ByT5-small  : /kaggle/input/notebooks/tatsuokoshida/2-8-dpc-starter-train-v1/byt5-akkadian-model/fulltrain_byt5-small_multitask
2026-03-21 11:37:19,110 | INFO |   Pool per model: beam x 4 + sample x 2
2026-03-21 11:37:19,111 | INFO |   MBR pool cap  : 32
2026-03-21 11:37:19,111 | INFO |   BF16 AMP      : False
2026-03-21 11:37:19,112 | INFO | 

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla P100-PCIE-16GB
GPU Mem  : 17.06 GB
BF16     : True
BF16 AMP : False
Attention: eager


Loading weights:   0%|          | 0/498 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-21 11:37:56,546 | INFO | [ByT5-large] attention backend: eager
2026-03-21 11:37:56,548 | INFO | [ByT5-large] 1,228,183,552 parameters
2026-03-21 11:37:56,549 | INFO | [ByT5-large] GPU mem used: 4.91 GB
2026-03-21 11:37:56,550 | INFO | [ByT5-large] BetterTransformer disabled to avoid GPU kernel incompatibility
2026-03-21 11:37:56,551 | INFO |   Bucket 0: 1 samples, len [20, 20]
2026-03-21 11:37:56,551 | INFO |   Bucket 1: 1 samples, len [20, 20]
2026-03-21 11:37:56,552 | INFO |   Bucket 2: 1 samples, len [23, 23]
2026-03-21 11:37:56,552 | INFO |   Bucket 3: 1 samples, len [38, 38]


[ByT5-large]:   0%|          | 0/4 [00:00<?, ?it/s]

...

[ByT5-small]:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-21 11:43:41,481 | INFO | Phase 4/4 - Pool merge + MBR selection


MBR:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-21 11:43:41,720 | INFO | ============================================================
2026-03-21 11:43:41,722 | INFO | Empty     : 0 (0.00%)
2026-03-21 11:43:41,723 | INFO | Len mean  : 33.2  median: 27.0  min: 23  max: 56
2026-03-21 11:43:41,724 | INFO |   ID 0: From <gap>,Kānesh-cludiṭšky:
2026-03-21 11:43:41,725 | INFO |   ID 1: From <gap> āli-kūn'sTABLE.
2026-03-21 11:43:41,726 | INFO |   ID 2: Since youplī-matrágṀŠINDEGAL's,MORTHPUZ.CB10:6345/278½⅔"
2026-03-21 11:43:41,727 | INFO |   ID 3: Send withour-lapìšṭābs,
2026-03-21 11:43:41,728 | INFO |   ID 3: Send withour-lapìšṭābs,
2026-03-21 11:43:41,729 | INFO | ============================================================
2026-03-21 11:43:41,736 | INFO | Saved -> /kaggle/working/submission.csv (4 rows)



Ensemble x MBR complete
Submission : /kaggle/working/submission.csv
Total rows : 4


## Notes for Kaggle submission
- 3つのモデルパスが Kaggle Notebook から見える状態で attach されていること。
- `max_input_length=1024`, `max_new_tokens=1024` のため、VRAM が厳しい場合は `batch_size` や候補数を先に落とすこと。
- OOM は `ensemble_mbr.log` に sample id 付きで記録される。
- フォールバックは `"<gap>"`。
- `BetterTransformer` は `cudaErrorNoKernelImageForDevice` 回避のため既定で無効。
- `attn_implementation="eager"` と `use_mixed_precision=False` を既定にして fused CUDA kernel を避ける。
- CUDA kernel incompatibilityを検出したモデルは自動でCPUへフォールバックする。
